# MORIARTY Geometry — Colab runner

Preregistered pipeline from [moriarty-geometry](https://github.com/moloodbahar/moriarty-geometry).

**Runtime:** GPU (T4 16 GB works for 7B models in bf16; use A100 for 14B).

**Order:** verify → behavioural pilot (H1 gate) → record activations → analyse.

Skip `01_reconstruct_confirmatory.py` — confirmatory episodes are already in `data/`.

In [ ]:
# Check GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Switch runtime: Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Optional: Hugging Face login (needed for gated models like Llama 3.1)
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
import os, pathlib

REPO = "https://github.com/moloodbahar/moriarty-geometry.git"
ROOT = pathlib.Path("/content/moriarty-geometry")

if not ROOT.exists():
    !git clone {REPO} {ROOT}
else:
    %cd {ROOT}
    !git pull

%cd {ROOT}
os.environ["PYTHONPATH"] = str(ROOT / "src")
print("Working directory:", ROOT)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Pre-flight checks
!python -m pytest -q tests
!python scripts/00_verify_reconstruction.py

## Configuration

Set `MODEL` to one of the four H1 observers (§3.3). Start with Qwen 7B on a T4.

The neutral registry is **DRAFT** — fine for development runs; set status to **FROZEN** in `data/neutral_registry.json` before confirmatory measurement.

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"   # H1 default
PILOT_FORMAT = "both"                 # native + decoupled (required for gate)
OUT_PILOT = "results/pilot_qwen7b"
OUT_ACT = "results/activations_qwen7b"
OUT_ANALYSIS = "results/analysis_qwen7b.json"

# All four H1 models (run one cell per model, or loop overnight):
H1_MODELS = [
    "meta-llama/Llama-3.1-8B-Instruct",   # gated — HF login required
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen2.5-14B-Instruct",          # needs A100 or similar
    "mistralai/Mistral-7B-Instruct-v0.3",
]

In [ ]:
# Stage 1 — behavioural pilot (~30–90 min on T4 for Qwen 7B, format=both)
import subprocess
subprocess.run([
    "python", "scripts/02_behavioural_pilot.py",
    "--model", MODEL,
    "--format", PILOT_FORMAT,
    "--out", OUT_PILOT,
    "--device", "cuda",
], check=True)

In [ ]:
# Quick look at clause-effect transfer (H1 gate: Δp ≥ 0.15)
import json
rows = json.load(open(f"{OUT_PILOT}/clause_effect_transfer.json"))
for r in rows:
    d = r.get("native:delta_present_minus_neutral") or r.get("decoupled:delta_present_minus_neutral")
    print(r["event_id"], r["type"], "Δp(native)=", r.get("native:delta_present_minus_neutral"), "Δp(decoupled)=", r.get("decoupled:delta_present_minus_neutral"))

In [ ]:
# Stage 2 — record activations (~15–30 min)
import subprocess
subprocess.run([
    "python", "scripts/03_record_activations.py",
    "--model", MODEL,
    "--format", "decoupled",
    "--out", OUT_ACT,
    "--device", "cuda",
], check=True)

In [ ]:
# Stage 3 — geometry + patching analysis
import subprocess, json
subprocess.run([
    "python", "scripts/04_analyse.py",
    "--activations", OUT_ACT,
    "--out", OUT_ANALYSIS,
], check=True)
print(json.dumps(json.load(open(OUT_ANALYSIS)), indent=2))

In [ ]:
# Optional: run all four H1 models (uncomment to use)
# import subprocess, pathlib
# for m in H1_MODELS:
#     slug = m.split("/")[-1].lower().replace(".", "")
#     out = f"results/pilot_{slug}"
#     print("===", m, "===")
#     subprocess.run(["python", "scripts/02_behavioural_pilot.py",
#                     "--model", m, "--format", "native", "--out", out, "--device", "cuda"], check=True)

In [ ]:
# Download results before the session ends
from google.colab import files
import shutil

shutil.make_archive("/content/moriarty-results", "zip", "results")
files.download("/content/moriarty-results.zip")